# 03_ml
Captured 2026-11-21 23:49 UTC

**Late-PO model beats the baseline on the time-split holdout:** PR-AUC 0.562 vs 0.455 (lift 0.107, bar 0.05, result: model adds value); recall in top 20% 0.347 vs 0.302. Scores feed gold.days_of_cover and disruption impact.


In [1]:
-- code: point-in-time features (only what was known at PO creation)
# src/ai/features.py
hist AS (
  SELECT p.po_number, p.po_line, count(h.po_number) AS vendor_receipts_365d,
         avg(CASE WHEN h.is_late THEN 0.0D ELSE 1.0D END) AS vendor_on_time_rate_365d
  FROM po p LEFT JOIN {catalog}.silver.po_line h
    ON h.vendor_id = p.vendor_id AND h.receipt_date < p.order_date
   AND h.receipt_date >= date_sub(p.order_date, 365)
  GROUP BY p.po_number, p.po_line),

used by training and scoring

In [2]:
-- code: time-split holdout, model vs baseline, no-claim rule
# src/ai/train_late_po.py
cutoff = as_of - timedelta(days=45)
df = spark.sql(feature_sql(a.catalog, f"receipt_date IS NOT NULL AND promised_date <= '{cutoff}'")).toPandas()
df["is_late"] = df["is_late"].astype(int)
split = df["order_date"].max() - timedelta(days=91)
train, test = df[df["order_date"] < split], df[df["order_date"] >= split]

model = Pipeline([
    ("prep", ColumnTransformer([("cat", OneHotEncoder(handle_unknown="ignore"), FEATURES_CAT)], remainder="passthrough")),
    ("clf", HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05, random_state=42)),
])
X_cols = FEATURES_NUM + FEATURES_CAT
mlflow.set_registry_uri("databricks-uc")
user = spark.sql("SELECT current_user() u").first().u
mlflow.set_experiment(f"/Users/{user}/shortage-radar-late-po")
with mlflow.start_run(run_name="late_po_hgb") as run:
    model.fit(train[X_cols], train["is_late"])
    p_model = model.predict_proba(test[X_cols])[:, 1]
    p_base = 1 - test["vendor_on_time_rate_365d"]
    m = {"train_rows": len(train), "test_rows": len(test), "test_late_rate": round(test["is_late"].mean(), 3),
         "pr_auc_model": round(average_precision_score(test["is_late"], p_model), 3),
         "pr_auc_baseline": round(average_precision_score(test["is_late"], p_base), 3),
         "roc_auc_model": round(roc_auc_score(test["is_late"], p_model), 3),
         "roc_auc_baseline": round(roc_auc_score(test["is_late"], p_base), 3),
         "recall_top20_model": recall_at_top(test["is_late"].values, p_model),
         "recall_top20_baseline": recall_at_top(test["is_late"].values, p_base.values)}
    m["pr_auc_lift"] = round(m["pr_auc_model"] - m["pr_auc_baseline"], 3)
    m["claim"] = "model adds value" if m["pr_auc_lift"] >= 0.

results in the next cell

In [3]:
-- training run (run_log/train_late_po.json)
-- src/ai/train_late_po.py

{"model_version": "6", "holdout_start": "2026-06-12", "train_rows": 36988, "test_rows": 3793, "test_late_rate": 0.342, "pr_auc_model": 0.562, "pr_auc_baseline": 0.455, "pr_auc_lift": 0.107, "recall_top20_model": 0.347, "recall_top20_baseline": 0.302, "claim": "model adds value"}

In [4]:
-- scoring run (run_log/score_po_risk.json)
-- src/ai/score_po_risk.py

{
  "task": "score_po_risk",
  "model_version": "6",
  "open_po_lines_scored": 4594,
  "mean_p_late": 0.355,
  "lines_p_late_over_0_5": 958,
  "mean_baseline_p_late": 0.336
}

In [5]:
-- serving endpoint state
databricks serving-endpoints get late-po-model

{"state": {"config_update": "NOT_UPDATING", "ready": "READY"}, "served": [["supply_chain.ml.late_po_model", "6"]]}

In [6]:
-- endpoint response: risky vs reliable vendor
databricks serving-endpoints query late-po-model

{"predictions": [[0.2650615609118987, 0.7349384390881013], [0.9392070752375183, 0.06079292476248162]]}

In [7]:
-- model output in the score table
SELECT ... FROM ml.po_risk_score

mean_model_p_late | mean_baseline_p_late | high_risk_lines | open_lines
--------------------------------------------------------------------------
0.355             | 0.336                | 958             | 4594      
(1 rows)

In [8]:
-- GenAI code: ai_query prompt and groundedness check
# src/ai/explain_impact.py
SELECT *, ai_query('{a.llm}', concat(
  'You are a pharmaceutical supply planner. In 2 sentences, explain the supply risk for this SKU to a planning lead and ',
  'name the single most useful next step. Use ONLY numbers and dates that appear in the data. Never mention price or revenue. Data: ',
  to_json(named_struct('sku_id', sku_id, 'drug', drug, 'region', region, 'api_id', api_id, 'single_source', single_source,
    'weekly_units', weekly_units, 'days_of_cover_now', cover_days_before, 'days_of_cover_without_vendor', cover_days_without_vendor,
    'days_to_qualify_alternate_source', requalification_days, 'projected_stockout_date', projected_stockout_date,
    'supply_gap_days', supply_gap_days)))) AS explanation,

# src/ai/explain_impact.py
def grounded(inputs, outputs) -> bool:
    """True when every number in the explanation appears in the source row."""
    found = {n.rstrip(".,") for n in NUM.findall(outputs)}  # drop trailing sentence punctuation
    return found <= inputs["allowed"]

result in the next cell

In [9]:
-- GenAI: ai_query explanations + groundedness (run_log/explain_impact.json)
-- src/ai/explain_impact.py

{"llm": "databricks-claude-haiku-5-5", "explanations": 25, "grounded": 25, "sample": {"sku_id": "SKU-0118", "explanation": "SKU-0118 (morphine, EU) is single-sourced from API M-API-015, and if that vendor fails, cover falls from 73.2 to 52.3 days, leaving a 127.7-day supply gap and a projected stockout on 2027-01-11. Qualifying an alternate source takes 180 days, which is longer than the current cover, so the next step is to start that qualification immediately."}}